# Olist revenue in euros — as it was handed over

A colleague wrote this notebook for a finance meeting in Europe. It reports Olist's revenue in **euros**, by month.

- **Revenue** is what customers paid: `payment_value` in `data/raw/order_payments.csv`, in Brazilian reais (BRL).
  An order belongs to the day and the month of its `order_purchase_timestamp` in `data/raw/orders.csv`.
- **The exchange rates** are the European Central Bank's, fetched once from the Frankfurter API and saved as
  `data/raw/api/frankfurter_eur.json`. How it was fetched: `DATA.md` and `scripts/fetch.py`.

A fact you can check anywhere: on every day from 2016 to 2018, **one Brazilian real was worth less than one euro**.

The cells under **The report** are your colleague's. They run without an error. Run them, read the numbers, then read
the README.

*(The World Bank files in `data/raw/api/worldbank/` and `toy_rates.json` are the lecture's. This lab does not need
them.)*

In [ ]:
import json
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 400)   # show every row of a result: a census is never "the top few"

# Anchor to the project folder (DS1, Block 1), then stand there: every path below is from the project folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Working in:", Path.cwd())   # must print this project's folder

con = duckdb.connect()             # an in-memory database; it reads the files in data/raw/ directly

## The report

### 1. The exchange rates, from the cached API response

In [ ]:
raw = json.load(open("data/raw/api/frankfurter_eur.json"))

rows = []
for day, values in raw["rates"].items():
    rows.append((day, values["BRL"]))

con.execute("CREATE OR REPLACE TABLE rates (date DATE, rate DOUBLE)")
con.executemany("INSERT INTO rates VALUES (?, ?)", rows)
print(len(rows), "days of rates, from", rows[0][0], "to", rows[-1][0])

### 2. Revenue per order, in reais

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE order_revenue AS
    SELECT o.order_id,
           CAST(o.order_purchase_timestamp AS DATE) AS order_date,
           SUM(p.payment_value)                     AS revenue_brl
    FROM 'data/raw/orders.csv' o
    JOIN 'data/raw/order_payments.csv' p USING (order_id)
    GROUP BY o.order_id, order_date
""")
con.sql("SELECT COUNT(*) AS orders, SUM(revenue_brl) AS revenue_brl FROM order_revenue").df()

### 3. Monthly revenue in euros

In [ ]:
con.sql("""
    SELECT date_trunc('month', r.order_date) AS month,
           COUNT(*)                          AS orders,
           SUM(r.revenue_brl * x.rate)       AS revenue_eur
    FROM order_revenue r
    JOIN rates x ON x.date = r.order_date
    GROUP BY month
    ORDER BY month
""").df()

### 4. Total revenue, in reais and in euros

In [ ]:
brl = con.sql("SELECT SUM(revenue_brl) FROM order_revenue").fetchone()[0]
eur = con.sql("""
    SELECT SUM(r.revenue_brl * x.rate)
    FROM order_revenue r
    JOIN rates x ON x.date = r.order_date
""").fetchone()[0]
print(f"Revenue: {brl:,.2f} BRL = {eur:,.2f} EUR")

### 5. Revenue in euros by weekday

In [ ]:
con.sql("""
    SELECT dayname(r.order_date)       AS weekday,
           COUNT(*)                    AS orders,
           SUM(r.revenue_brl * x.rate) AS revenue_eur
    FROM order_revenue r
    JOIN rates x ON x.date = r.order_date
    GROUP BY weekday, isodow(r.order_date)
    ORDER BY isodow(r.order_date)
""").df()

---

# Your work starts here

Work top to bottom. Paste what section A shows you into `DIAGNOSIS.md`, part 3, **before** you change anything.

> **A known issue, disclosed (the README says it too).** The ECB publishes rates on its business days only: no
> weekends, no ECB holidays. **625 of the 2,652 paid orders** were placed on such a day, so they have no rate for
> their own date. You do not need to diagnose this. Handle it: your monthly query (section D) must keep those
> orders, and your check (section E) must count any order that ends up with no euro value.
>
> *Only if the instructor announces the single-table route:* the monthly query and that check move to Homework 2.
> Your route is at the end of the notebook: A, B, C, then R1 to R4.

## A. Read the document before its numbers

The report used one field of the JSON, `rates`, and one key inside it, `BRL`. The document has more to say.
Run these two cells (supplied), and write one sentence in a comment under each: what did it tell you?

In [ ]:
# 1. Everything in the response except the rates themselves.
{key: value for key, value in raw.items() if key != "rates"}

In [ ]:
# 2. One day's entry, exactly as the document has it.
raw["rates"]["2017-01-02"]

The known issue, counted (supplied): **the three counts** for the report's join, `order_revenue` to `rates` on the
date — rows in the left table, rows in the result, distinct `order_id` in the result. This is the evidence for the
disclosed issue; paste it into part 3 of your note too.

In [ ]:
con.sql("""
    SELECT (SELECT COUNT(*) FROM order_revenue)                              AS left_rows,
           COUNT(*)                                                          AS result_rows,
           COUNT(DISTINCT r.order_id)                                        AS distinct_orders_in_result
    FROM order_revenue r
    JOIN rates x ON x.date = r.order_date
""").df()

## B. The rates table, the right way round

Write this from the empty cell. Build a table `eur_rates` with **one row per date per currency** — every currency the
document has, not only `BRL`:

| column | meaning |
|---|---|
| `date` | the ECB business day |
| `currency` | the currency's code, taken from the keys inside `rates` |
| `eur_per_unit` | how many **euros** one unit of that currency buys on that day |

Take the direction from the document's own fields (section A), not from what the key names suggest. Use a Python loop,
as the lecture showed on its toy; this document has one more level than the toy had. The last two lines are supplied.

In [ ]:
# Your loop. It builds a list of (date, currency, eur_per_unit) tuples called rows.


# con.execute("CREATE OR REPLACE TABLE eur_rates (date DATE, currency VARCHAR, eur_per_unit DOUBLE)")
# con.executemany("INSERT INTO eur_rates VALUES (?, ?, ?)", rows)

## C. Prove the direction: one order, by hand

A table can be wrong in a way no total will show you. So convert **one order** twice, independently:

1. **By hand.** Pick one order placed on a weekday. Write, in comments: its `order_id`, its date, what it paid in
   reais, and that date's `BRL` value copied from the JSON itself (section A shows you how to look one day up). Then
   the conversion, written out: euros = ... Say in words what the JSON's number means.
2. **By your table.** Convert the same order with `eur_rates`, in SQL.

Print both. They must agree **to the cent**: `assert abs(by_hand - by_table) < 0.005`. Never `==` on two money
figures computed in different ways; the last binary digit of a float can differ when the cents do not. This cell is
what your note's part 5 is built on.

In [ ]:
# By hand, in comments; then by the table, in SQL; print both.

## D. Monthly revenue in euros, to the finance team's definition

The finance team's rule: **convert each order at the average of the ECB's daily EUR-per-BRL rates in the month the
order was placed.** Write the query from the empty cell: one row per month with `month`, `orders`, `revenue_brl`,
`revenue_eur`. Every order in `order_revenue` must be in it exactly once.

In [ ]:
# Your query: monthly revenue in reais and in euros, at the month's average rate.

## E. The check

Write it so that it would fail if either of today's problems came back:

1. **Count the conversions that are missing**, not the rows that joined: how many orders have no euro value? Must be
   0. This is the check for the disclosed issue: a join that lost the 625 orders, or kept them with no rate, fails it.
2. **Count the orders** in your monthly table, and **add up its reais**. The orders must equal those in
   `order_revenue`; the reais must equal its total **to the cent** — `abs(a - b) < 0.005`, never `==` on sums of
   money.
3. **The range bound** (plausibility only): euros ÷ reais for the whole period is a revenue-weighted average of the
   monthly rates, so it must lie between the lowest and the highest monthly rate. Print all three numbers. The
   lecture showed, on the World Bank's population, why a figure for the whole can pass this bound and still be
   wrong; section C is the proof.

In [ ]:
# The missing-conversion count, the order count, and the range bound. Assert the first two.

## F. The corrected total, and the table kept (the last cell is supplied)

Print the report's line again, from your monthly table: `Revenue: ... BRL = ... EUR`.

In [ ]:
# The corrected total.

In [ ]:
# Supplied. Uncomment and run once eur_rates exists.
# os.makedirs("data/silver", exist_ok=True)
# con.sql("COPY (SELECT * FROM eur_rates ORDER BY date, currency) TO 'data/silver/eur_rates.csv' (HEADER)")
# print("wrote data/silver/eur_rates.csv:", con.sql("SELECT COUNT(*) FROM eur_rates").fetchone()[0], "rows")

## Stretch (only when A to F are done)

**(a)** The same revenue in US dollars, from the second currency in your table: each month's euros divided by that
month's average EUR-per-USD rate. Then the identity that ties the two currencies together, checked on three dates of
your choice: reais per dollar = (the JSON's `BRL` value) ÷ (the JSON's `USD` value) = your `eur_per_unit` for USD ÷
your `eur_per_unit` for BRL.

**(b)** Daily instead of monthly: convert each order at the rate of its own day, or, for a day with no rate, the last
rate before it. DuckDB's `ASOF JOIN` does this in one line: `FROM order_revenue r ASOF JOIN brl_rates x ON
r.order_date >= x.date`, where `brl_rates` is your table's BRL rows. Compare the euro total with section D's.
Which definition should the note carry?

**(c)** Write one request yourself — the lab never needs it, and it needs the network. Ask Frankfurter for **one
day**, say 2017-01-02, with the same base and symbols as the cache: the date goes in the path
(`https://api.frankfurter.dev/v1/2017-01-02`), `base` and `symbols` go in the query string (build it with
`urllib.parse.urlencode`). Send a `User-Agent` header that names your program — Frankfurter answers Python's default
one with `403 Forbidden` — and a `timeout`. Check the status is 200. Save the answer under `output/` (Git ignores it;
never over `data/raw/`), then compare its rates with that day in the cache. Put the whole thing behind a switch,
`LIVE = False`, so the notebook stays offline unless you turn it on.

In [ ]:
# Stretch (a), (b) and (c).

---

# Only if the instructor announces it: the single-table route

**Skip this section unless the instructor says, at the start of the lab, that it is on.** If it is, this is your
whole route, in order:

1. Sections **A, B and C**, as everyone does them. They do not change: the rates table and the hand calculation are
   the lab.
2. **R1, R2, R3** below, **instead of section D**. Each is one query, from an empty cell, on one table. Write how
   many rows you expect, and why, before you run it.
3. **R4** below, **instead of section E**: the check for this route.
4. **The last cell of section F** (supplied), which keeps your `eur_rates` in `data/silver/`. Skip F's corrected
   total and the stretch.
5. `DIAGNOSIS.md` (the README's *single-table route* section says what goes in each part).

The monthly query, its check and the corrected total move to Homework 2, whose first repair is that same query.

**R1 — `GROUP BY`.** For each year, how many days have a rate in the report's `rates` table, and what were the
lowest and the highest values? (That table holds the JSON's `BRL` values as they are: reais for one euro.)

In [ ]:
# R1

**R2 — `HAVING`.** Which months had more than 180 orders? Month and number of orders, from `data/raw/orders.csv`,
busiest first.

In [ ]:
# R2

**R3 — a NULL-safe filter.** How many orders were **not** delivered late? An order is late when
`order_delivered_customer_date` is after `order_estimated_delivery_date`. An order with no delivery date was never
delivered, so it was not delivered late: it counts.

In [ ]:
# R3

**R4 — the check for this route.** Two checks on your `eur_rates`, each asserted and printed:

1. **Its grain.** One row per date per currency: no `(date, currency)` pair appears twice (a `GROUP BY` with a
   `HAVING`, as in R2), and the rows equal the days times the currencies.
2. **Its direction, as plausibility.** A real was worth less than a euro on every day of the period, so every `BRL`
   row's `eur_per_unit` must be below 1. A table built the wrong way round fails this. Passing it does not prove the
   table right: section C's hand calculation is the proof.

In [ ]:
# R4: the grain check and the direction check, asserted and printed.